# 📖 Notebook 1: Bid Processing & Concurrency

Bidding is the heart of an online auction. When two people bid at the same time, bad things can happen — both bids get accepted, the wrong person wins, or bids silently disappear.

In this notebook, we'll **break the system on purpose** to see these race conditions, then fix them with three different approaches.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why naive read-then-write creates race conditions
- How **row-level locking** (pessimistic) serializes bid processing
- How **optimistic concurrency control** (OCC) avoids locks entirely
- How **Redis Lua scripts** provide atomic compare-and-set operations
- The tradeoffs between each approach

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/online-auction
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `auction_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import threading

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "auction_demo",
    "user": "demo",
    "password": "demo"
}

# Redis connection settings
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    """Create a new PostgreSQL connection."""
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    """Create a new Redis client."""
    return redis.Redis(**REDIS_CONFIG)

# Test both connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker-compose up -d")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker-compose up -d")

In [ ]:
# Helper: reset an auction to a known state for each experiment

def reset_auction(auction_id, starting_max=1000.00):
    """Reset an auction's max bid so we can re-run experiments cleanly."""
    conn = get_db_connection()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(
        "UPDATE auctions SET max_bid_amount = %s, max_bid_user_id = NULL WHERE id = %s",
        (starting_max, auction_id)
    )
    # Clean up any test bids
    cur.execute("DELETE FROM bids WHERE auction_id = %s AND amount >= 5000", (auction_id,))
    conn.close()
    print(f"🔄 Auction {auction_id} reset to max_bid = ${starting_max:.2f}")

def show_auction(auction_id):
    """Display the current state of an auction."""
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute(
        """SELECT a.id, i.name, a.max_bid_amount, a.max_bid_user_id, a.status
           FROM auctions a JOIN items i ON a.item_id = i.id
           WHERE a.id = %s""",
        (auction_id,)
    )
    row = cur.fetchone()
    conn.close()
    if row:
        print(f"🏷️  Auction #{row[0]}: {row[1]}")
        print(f"   Current highest bid: ${row[2]:,.2f}")
        print(f"   Leading bidder: User {row[3] or 'None'}")
        print(f"   Status: {row[4]}")
    return row

# Let's see a sample auction
print("📋 Sample auction from our database:\n")
show_auction(1)

---
## 🐛 The Problem: Race Conditions

Imagine this scenario:

1. The current highest bid on a guitar is **$1,000**
2. **User A** wants to bid **$5,000** (a big jump!)
3. **User B** wants to bid **$1,500** at almost the same time

With a naive approach (read the max, check if your bid is higher, then write), this can happen:

```
Time    User A                          User B
─────   ──────────────────────────      ──────────────────────────
T1      Read max_bid → $1,000
T2                                      Read max_bid → $1,000
T3      $5,000 > $1,000? ✅ Accept
T4      Write max_bid = $5,000
T5                                      $1,500 > $1,000? ✅ Accept
T6                                      Write max_bid = $1,500  ← BUG!
```

User B's $1,500 bid **overwrites** User A's $5,000 bid! The auction now shows $1,500 as the highest bid. User A got robbed.

Let's reproduce this bug with real code.

In [ ]:
# ❌ BROKEN: Naive bid placement (read-then-write, no protection)

def place_bid_naive(auction_id, user_id, amount):
    """
    Place a bid WITHOUT any concurrency protection.
    This is how a beginner might write it — and it's broken.
    """
    conn = get_db_connection()
    cur = conn.cursor()

    # Step 1: Read the current max bid
    cur.execute("SELECT max_bid_amount FROM auctions WHERE id = %s", (auction_id,))
    current_max = float(cur.fetchone()[0])

    # Simulate network delay — this makes the race condition more likely
    time.sleep(0.1)

    # Step 2: Check if our bid is higher
    if amount > current_max:
        # Step 3: Write the new max bid
        cur.execute(
            "UPDATE auctions SET max_bid_amount = %s, max_bid_user_id = %s WHERE id = %s",
            (amount, user_id, auction_id)
        )
        cur.execute(
            "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'accepted')",
            (auction_id, user_id, amount)
        )
        conn.commit()
        conn.close()
        return {"status": "accepted", "amount": amount, "user_id": user_id}
    else:
        cur.execute(
            "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'rejected')",
            (auction_id, user_id, amount)
        )
        conn.commit()
        conn.close()
        return {"status": "rejected", "amount": amount, "user_id": user_id}

In [ ]:
# Let's trigger the race condition!
# We use auction 16 (Dyson vacuum, starts at $300, no bids yet)

AUCTION_ID = 16
reset_auction(AUCTION_ID, starting_max=1000.00)
print()

results = {}

def bid_thread(name, auction_id, user_id, amount):
    """Run a bid in a separate thread to simulate concurrent users."""
    result = place_bid_naive(auction_id, user_id, amount)
    results[name] = result

# User A bids $5,000 — this SHOULD be the winner
# User B bids $1,500 — this should be rejected (lower than $5,000)
thread_a = threading.Thread(target=bid_thread, args=("User A", AUCTION_ID, 10, 5000.00))
thread_b = threading.Thread(target=bid_thread, args=("User B", AUCTION_ID, 20, 1500.00))

# Start both threads at almost the same time
thread_a.start()
thread_b.start()
thread_a.join()
thread_b.join()

print("📊 Results:")
for name, result in results.items():
    emoji = "✅" if result["status"] == "accepted" else "❌"
    print(f"   {emoji} {name} bid ${result['amount']:,.2f} → {result['status']}")

print()
print("📋 Final auction state:")
show_auction(AUCTION_ID)

print()
print("⚠️  Both bids were ACCEPTED! The $1,500 bid overwrote the $5,000 bid.")
print("   This is the classic read-then-write race condition.")

---
## 🔒 Fix 1: Row-Level Locking (Pessimistic Approach)

The simplest fix is to **lock the auction row** while processing a bid. This forces concurrent bids to wait in line.

### How It Works
1. `BEGIN` a transaction
2. `SELECT ... FOR UPDATE` — this locks the auction row. Any other transaction trying to read the same row will **wait** until we're done.
3. Check if the new bid is higher than the current max
4. If yes, update the auction and insert the bid
5. `COMMIT` — this releases the lock

### Why Lock the Auction Row (Not All Bid Rows)?
An earlier approach in the source material locks **all bid rows** (`SELECT * FROM bids WHERE auction_id = ? FOR UPDATE`). This is bad because:
- As bids grow, you lock more and more rows
- `FOR UPDATE` only locks **existing** rows, not new inserts
- Performance degrades over time

Instead, we lock **one row** on the `auctions` table. This is fast and scales well.

In [ ]:
# ✅ FIXED: Bid placement with row-level locking (pessimistic concurrency)

def place_bid_with_lock(auction_id, user_id, amount):
    """
    Place a bid using SELECT ... FOR UPDATE to lock the auction row.
    This guarantees only one bid is processed at a time for this auction.
    """
    conn = get_db_connection()
    cur = conn.cursor()

    try:
        # Step 1: Lock the auction row and read the current max bid
        # FOR UPDATE tells Postgres: "Lock this row — nobody else can read or
        # modify it until I commit or rollback."
        cur.execute(
            "SELECT max_bid_amount FROM auctions WHERE id = %s FOR UPDATE",
            (auction_id,)
        )
        current_max = float(cur.fetchone()[0])

        # Simulate network delay — even with this, the lock keeps us safe
        time.sleep(0.1)

        # Step 2: Check and write
        if amount > current_max:
            cur.execute(
                "UPDATE auctions SET max_bid_amount = %s, max_bid_user_id = %s WHERE id = %s",
                (amount, user_id, auction_id)
            )
            cur.execute(
                "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'accepted')",
                (auction_id, user_id, amount)
            )
            conn.commit()
            conn.close()
            return {"status": "accepted", "amount": amount, "user_id": user_id}
        else:
            cur.execute(
                "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'rejected')",
                (auction_id, user_id, amount)
            )
            conn.commit()
            conn.close()
            return {"status": "rejected", "amount": amount, "user_id": user_id}
    except Exception as e:
        conn.rollback()
        conn.close()
        return {"status": "error", "amount": amount, "user_id": user_id, "error": str(e)}

In [ ]:
# Test: Same scenario, but now with locking

reset_auction(AUCTION_ID, starting_max=1000.00)
print()

results = {}

def bid_thread_locked(name, auction_id, user_id, amount):
    result = place_bid_with_lock(auction_id, user_id, amount)
    results[name] = result

thread_a = threading.Thread(target=bid_thread_locked, args=("User A", AUCTION_ID, 10, 5000.00))
thread_b = threading.Thread(target=bid_thread_locked, args=("User B", AUCTION_ID, 20, 1500.00))

thread_a.start()
thread_b.start()
thread_a.join()
thread_b.join()

print("📊 Results with row-level locking:")
for name, result in results.items():
    emoji = "✅" if result["status"] == "accepted" else "❌"
    print(f"   {emoji} {name} bid ${result['amount']:,.2f} → {result['status']}")

print()
print("📋 Final auction state:")
show_auction(AUCTION_ID)

print()
print("🎉 User A's $5,000 bid wins. User B's $1,500 is correctly rejected.")
print("   The lock forced User B to wait until User A's transaction completed.")

### Tradeoffs of Row-Level Locking

| Pro | Con |
|-----|-----|
| Simple to implement | Bids are serialized — one at a time per auction |
| Strong consistency guaranteed | Waiting threads block (wasted resources) |
| Works with any SQL database | Lock contention increases with popularity |

For a hot auction with hundreds of bids per second, this approach can become a bottleneck. The next approach avoids locking entirely.

---
## ⚡ Fix 2: Optimistic Concurrency Control (OCC)

**Key insight**: Bid conflicts are actually **rare**. Most bids don't happen at the exact same millisecond. So instead of locking (which is expensive), we can be **optimistic**:

1. Read the current max bid (no lock!)
2. Try to update the auction, **but only if the max bid hasn't changed** since we read it
3. If someone else updated it first, our update touches 0 rows — we detect this and **retry**

The magic is in this SQL:
```sql
UPDATE auctions
SET max_bid_amount = $new_bid
WHERE id = $auction_id AND max_bid_amount = $original_max
```

The `AND max_bid_amount = $original_max` is the **version check**. If someone else changed it between our read and write, this `WHERE` clause won't match, and zero rows get updated.

In [ ]:
# ✅ FIXED: Bid placement with Optimistic Concurrency Control

def place_bid_occ(auction_id, user_id, amount, max_retries=5):
    """
    Place a bid using Optimistic Concurrency Control.
    No locks! Instead, we detect conflicts and retry.
    """
    for attempt in range(max_retries):
        conn = get_db_connection()
        conn.autocommit = False
        cur = conn.cursor()

        try:
            # Step 1: Read the current max bid (no lock)
            cur.execute("SELECT max_bid_amount FROM auctions WHERE id = %s", (auction_id,))
            current_max = float(cur.fetchone()[0])

            # Is our bid even high enough?
            if amount <= current_max:
                cur.execute(
                    "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'rejected')",
                    (auction_id, user_id, amount)
                )
                conn.commit()
                conn.close()
                return {"status": "rejected", "amount": amount, "user_id": user_id, "attempts": attempt + 1}

            # Step 2: Try to update — but ONLY if max_bid hasn't changed
            cur.execute(
                """UPDATE auctions
                   SET max_bid_amount = %s, max_bid_user_id = %s
                   WHERE id = %s AND max_bid_amount = %s""",
                (amount, user_id, auction_id, current_max)
            )

            # Check if the update actually changed a row
            if cur.rowcount == 1:
                # Success! Nobody changed the max bid between our read and write.
                cur.execute(
                    "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'accepted')",
                    (auction_id, user_id, amount)
                )
                conn.commit()
                conn.close()
                return {"status": "accepted", "amount": amount, "user_id": user_id, "attempts": attempt + 1}
            else:
                # Conflict! Someone else updated the max bid. Rollback and retry.
                conn.rollback()
                conn.close()
                print(f"   🔄 User {user_id}: Conflict on attempt {attempt + 1}, retrying...")
                continue

        except Exception as e:
            conn.rollback()
            conn.close()
            return {"status": "error", "amount": amount, "user_id": user_id, "error": str(e)}

    return {"status": "failed_after_retries", "amount": amount, "user_id": user_id}

In [ ]:
# Test: OCC with concurrent bids

reset_auction(AUCTION_ID, starting_max=1000.00)
print()

results = {}

def bid_thread_occ(name, auction_id, user_id, amount):
    result = place_bid_occ(auction_id, user_id, amount)
    results[name] = result

thread_a = threading.Thread(target=bid_thread_occ, args=("User A", AUCTION_ID, 10, 5000.00))
thread_b = threading.Thread(target=bid_thread_occ, args=("User B", AUCTION_ID, 20, 1500.00))

thread_a.start()
thread_b.start()
thread_a.join()
thread_b.join()

print("📊 Results with OCC:")
for name, result in results.items():
    emoji = "✅" if result["status"] == "accepted" else "❌"
    attempts = result.get("attempts", "?")
    print(f"   {emoji} {name} bid ${result['amount']:,.2f} → {result['status']} (attempts: {attempts})")

print()
print("📋 Final auction state:")
show_auction(AUCTION_ID)

print()
print("🎉 Correct result achieved without any locks!")
print("   If there was a conflict, the losing transaction simply retried.")

In [ ]:
# Stress test: 10 concurrent bidders, each bidding a different amount

reset_auction(AUCTION_ID, starting_max=1000.00)
print()
print("🏁 Stress test: 10 users bidding at the same time...\n")

stress_results = {}

def stress_bid(user_id, amount):
    result = place_bid_occ(AUCTION_ID, user_id, amount)
    stress_results[user_id] = result

threads = []
bids = [
    (1, 5100), (2, 5200), (3, 5300), (4, 5400), (5, 5500),
    (6, 5600), (7, 5700), (8, 5800), (9, 5900), (10, 6000),
]

for user_id, amount in bids:
    t = threading.Thread(target=stress_bid, args=(user_id, amount))
    threads.append(t)

# Start all 10 threads at once
for t in threads:
    t.start()
for t in threads:
    t.join()

print("\n📊 Stress test results:")
for user_id in sorted(stress_results.keys()):
    r = stress_results[user_id]
    emoji = "✅" if r["status"] == "accepted" else "❌"
    print(f"   {emoji} User {user_id:>2} bid ${r['amount']:>8,.2f} → {r['status']:<10} (attempts: {r.get('attempts', '?')})")

print()
print("📋 Final auction state:")
show_auction(AUCTION_ID)

print()
print("💡 Notice how some users needed multiple attempts (retries) due to conflicts.")
print("   But the final result is always correct — the highest bid wins.")

### Tradeoffs of OCC

| Pro | Con |
|-----|-----|
| No locks — reads never block | Retries needed on conflict |
| Higher throughput under low contention | Under very high contention, many retries |
| Simple to implement | Wasted work on failed attempts |

**OCC is ideal for auctions** because bid conflicts are relatively rare. Most of the time, bids arrive seconds apart, not at the exact same moment.

---
## 🚀 Fix 3: Redis Atomic Compare-and-Set (Lua Script)

What if we move the "current max bid" into **Redis** instead of reading it from Postgres each time?

Redis is single-threaded, so operations run one at a time. By using a **Lua script**, we can read the current max, compare, and set the new max as **one atomic operation** — no locks, no retries.

### How It Works
1. Store the current max bid for each auction in Redis: `auction:{id}:max_bid`
2. When a bid comes in, run a Lua script that:
   - Reads the current max from Redis
   - Compares the new bid against it
   - If higher, updates Redis and returns `1` (accepted)
   - If not, returns `0` (rejected)
3. If accepted, write the bid to Postgres (permanent storage)

### Why Lua?
Redis `MULTI/EXEC` (transactions) can't read a value and conditionally write based on it. Lua scripts run atomically inside Redis — they can do read-modify-write in one step.

In [ ]:
# ✅ FIXED: Bid placement with Redis Lua script (atomic compare-and-set)

# This Lua script runs INSIDE Redis — it's atomic (no other command can run
# between the GET and SET). Redis is single-threaded, so this is safe.
COMPARE_AND_SET_LUA = """
local current_max = tonumber(redis.call('GET', KEYS[1]) or '0')
local proposed_bid = tonumber(ARGV[1])
local user_id = ARGV[2]

if proposed_bid > current_max then
    redis.call('SET', KEYS[1], proposed_bid)
    redis.call('SET', KEYS[1] .. ':user', user_id)
    return 1
else
    return 0
end
"""

def place_bid_redis(auction_id, user_id, amount):
    """
    Place a bid using Redis for the fast atomic check,
    then persist to Postgres for durability.
    """
    r = get_redis_client()
    cache_key = f"auction:{auction_id}:max_bid"

    # Step 1: Atomic compare-and-set in Redis
    accepted = r.eval(COMPARE_AND_SET_LUA, 1, cache_key, str(amount), str(user_id))

    # Step 2: Write to Postgres for permanent storage
    conn = get_db_connection()
    conn.autocommit = True
    cur = conn.cursor()

    status = "accepted" if accepted == 1 else "rejected"
    cur.execute(
        "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, %s)",
        (auction_id, user_id, amount, status)
    )

    if accepted == 1:
        cur.execute(
            "UPDATE auctions SET max_bid_amount = %s, max_bid_user_id = %s WHERE id = %s",
            (amount, user_id, auction_id)
        )

    conn.close()
    return {"status": status, "amount": amount, "user_id": user_id}

In [ ]:
# Test: Redis atomic approach with concurrent bids

reset_auction(AUCTION_ID, starting_max=1000.00)

# Seed the Redis cache with the current max bid
r = get_redis_client()
r.set(f"auction:{AUCTION_ID}:max_bid", "1000.00")
print(f"🔄 Redis seeded: auction:{AUCTION_ID}:max_bid = 1000.00\n")

results = {}

def bid_thread_redis(name, auction_id, user_id, amount):
    result = place_bid_redis(auction_id, user_id, amount)
    results[name] = result

thread_a = threading.Thread(target=bid_thread_redis, args=("User A", AUCTION_ID, 10, 5000.00))
thread_b = threading.Thread(target=bid_thread_redis, args=("User B", AUCTION_ID, 20, 1500.00))

thread_a.start()
thread_b.start()
thread_a.join()
thread_b.join()

print("📊 Results with Redis Lua:")
for name, result in results.items():
    emoji = "✅" if result["status"] == "accepted" else "❌"
    print(f"   {emoji} {name} bid ${result['amount']:,.2f} → {result['status']}")

print()
print("📋 Final auction state:")
show_auction(AUCTION_ID)

# Show what Redis has
print()
print(f"📦 Redis state:")
print(f"   auction:{AUCTION_ID}:max_bid = {r.get(f'auction:{AUCTION_ID}:max_bid')}")
print(f"   auction:{AUCTION_ID}:max_bid:user = {r.get(f'auction:{AUCTION_ID}:max_bid:user')}")

# Cleanup Redis keys
r.delete(f"auction:{AUCTION_ID}:max_bid", f"auction:{AUCTION_ID}:max_bid:user")

### Tradeoffs of Redis Lua

| Pro | Con |
|-----|-----|
| Extremely fast (~1ms) | Two systems to keep in sync (Redis + Postgres) |
| No locks, no retries | If Redis and Postgres disagree, which is correct? |
| Scales independently of DB | Redis is in-memory — data can be lost on crash |

### The Consistency Challenge

The big question: **what happens if Redis says "accepted" but the Postgres write fails?**

Options:
1. **Accept Redis as source of truth** during the auction, write to Postgres async
2. **Write to Postgres first**, update Redis if DB succeeds, invalidate cache if it fails
3. **Use Redis only as a fast check**, then do the real write with OCC in Postgres

There's no perfect answer — this is why distributed consistency is one of the hardest problems in system design!

---
## 📊 Comparing All Three Approaches

Let's benchmark all three with a burst of 20 concurrent bids.

In [ ]:
import statistics

def benchmark(bid_fn, label, num_bids=20, setup_redis=False):
    """Run concurrent bids and measure total time and correctness."""
    reset_auction(AUCTION_ID, starting_max=1000.00)

    if setup_redis:
        r = get_redis_client()
        r.set(f"auction:{AUCTION_ID}:max_bid", "1000.00")

    results = {}
    def run_bid(idx):
        amount = 1100 + (idx * 100)
        start = time.time()
        result = bid_fn(AUCTION_ID, idx + 1, amount)
        elapsed = (time.time() - start) * 1000
        result["latency_ms"] = elapsed
        results[idx] = result

    threads = [threading.Thread(target=run_bid, args=(i,)) for i in range(num_bids)]

    start_time = time.time()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    total_time = (time.time() - start_time) * 1000

    # Check correctness — the highest amount should be the max bid
    expected_max = 1100 + (num_bids - 1) * 100
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("SELECT max_bid_amount FROM auctions WHERE id = %s", (AUCTION_ID,))
    actual_max = float(cur.fetchone()[0])
    conn.close()

    accepted_count = sum(1 for r in results.values() if r["status"] == "accepted")
    latencies = [r["latency_ms"] for r in results.values()]

    if setup_redis:
        r = get_redis_client()
        r.delete(f"auction:{AUCTION_ID}:max_bid", f"auction:{AUCTION_ID}:max_bid:user")

    print(f"\n{'='*50}")
    print(f"📊 {label}")
    print(f"{'='*50}")
    correct = "✅ CORRECT" if actual_max == expected_max else f"❌ WRONG (expected {expected_max}, got {actual_max})"
    print(f"   Correctness:     {correct}")
    print(f"   Total time:      {total_time:.0f} ms")
    print(f"   Avg latency:     {statistics.mean(latencies):.0f} ms")
    print(f"   Bids accepted:   {accepted_count}/{num_bids}")

# Run all three benchmarks
benchmark(place_bid_naive,     "Naive (BROKEN — no protection)")
benchmark(place_bid_with_lock, "Row-Level Locking (Pessimistic)")
benchmark(place_bid_occ,       "Optimistic Concurrency Control")
benchmark(place_bid_redis,     "Redis Lua (Atomic Compare-and-Set)", setup_redis=True)

print("\n" + "="*50)
print("💡 Key takeaways:")
print("   - Naive is fast but WRONG — race conditions corrupt data")
print("   - Locking is correct but slow — bids are serialized")
print("   - OCC is correct and fast — retries handle rare conflicts")
print("   - Redis Lua is the fastest — but adds cross-system complexity")

---
## 🧠 Summary

| Approach | Consistency | Performance | Complexity | Best For |
|----------|------------|-------------|------------|----------|
| Naive (broken) | ❌ None | ⚡ Fast | Low | Nothing — it's broken |
| Row Locking | ✅ Strong | 🐢 Slow under contention | Low | Simple apps, low traffic |
| OCC | ✅ Strong | ⚡ Fast (retries on conflict) | Medium | Most auction systems |
| Redis Lua | ✅ Atomic in Redis | ⚡⚡ Very fast | High | High-throughput systems |

### In a Real System
- **OCC on the auction row** is the recommended approach for most cases
- **Redis Lua** is used when you need extreme throughput and accept the complexity of keeping Redis and Postgres in sync
- A **message queue** (Kafka) in front of the bid service adds durability — bids are never lost even if the service crashes

### What's Next
- **Notebook 2**: Auction lifecycle — creating, ending, and managing auction state
- **Notebook 3**: Real-time notifications — pushing bid updates to all watchers instantly